# Demo 06. Quadrature: Newton-Cotes, Composite Rules, and Adaptive Simpson

**Module 4** (quadrature), sections 4.1 and 4.4.

An interpolatory quadrature rule integrates the polynomial interpolant of the integrand. Fitting the interpolant on equispaced nodes gives the Newton-Cotes rules: midpoint, trapezoid, and Simpson. Applying a rule on many small panels gives a composite rule whose error decays at a fixed rate. When the integrand has a localized feature, a uniform panel width is wasteful; adaptive Simpson refines only where an error estimate demands it. This demo measures the composite convergence rates and shows the adaptive method concentrating work where it is needed.

In [ ]:
# --- Colab / Jupyter setup ------------------------------------------------
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, Dropdown, FloatSlider, IntSlider

%matplotlib inline
plt.rcParams["figure.figsize"] = (9, 5)

## Rules and their orders

On a panel of width $h$ the local errors are $O(h^3)$ for midpoint and trapezoid and $O(h^5)$ for Simpson. Summing over $n=(b-a)/h$ panels lowers each power by one, so the composite errors are

$$ E_{\text{trap}},\,E_{\text{mid}} = O(h^2), \qquad E_{\text{Simpson}} = O(h^4). $$

Simpson gains two extra orders because it integrates a quadratic through the panel and is in fact exact for cubics: its degree of exactness is three. On log-log axes the composite error against the number of panels is a straight line whose slope is the negative of the order.

In [ ]:
# ---------------------------------------------------------------------------
# Composite Newton-Cotes rules on [a, b] with n panels
# ---------------------------------------------------------------------------
# Each rule samples the integrand on a uniform grid and forms a weighted sum.
# The weights are exactly those that integrate the local interpolating
# polynomial: constants for midpoint, a line for trapezoid, a parabola for
# Simpson.

def composite_midpoint(f, a, b, n):
    """Composite midpoint rule. Samples panel centers. Order 2."""
    h = (b - a) / n
    x = a + (np.arange(n) + 0.5) * h        # panel midpoints
    return h * np.sum(f(x))

def composite_trapezoid(f, a, b, n):
    """Composite trapezoid rule. Order 2."""
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h * (0.5 * y[0] + np.sum(y[1:-1]) + 0.5 * y[-1])

def composite_simpson(f, a, b, n):
    """Composite Simpson rule. Requires an even number of panels. Order 4."""
    if n % 2 == 1:
        n += 1                              # Simpson pairs panels; force even
    x = np.linspace(a, b, n + 1)
    y = f(x)
    h = (b - a) / n
    return h / 3.0 * (y[0] + y[-1] + 4.0 * np.sum(y[1:-1:2]) + 2.0 * np.sum(y[2:-1:2]))

In [ ]:
# ---------------------------------------------------------------------------
# Measure the composite convergence rate
# ---------------------------------------------------------------------------
# Test integral: exp(x) on [0, 1], whose exact value is e - 1. We halve the
# panel width repeatedly and fit a line to log(error) vs log(n). The fitted
# slope is the empirical order of the rule.
import math

def f_smooth(x): return np.exp(x)
A, B = 0.0, 1.0
I_EXACT = math.e - 1.0

def show_convergence():
    """Plot composite error vs number of panels and report the fitted order."""
    ns = 2 ** np.arange(1, 9)               # 2, 4, ..., 256 panels
    rules = {
        "trapezoid": composite_trapezoid,
        "midpoint":  composite_midpoint,
        "Simpson":   composite_simpson,
    }
    plt.figure()
    for name, rule in rules.items():
        err = np.array([abs(rule(f_smooth, A, B, n) - I_EXACT) for n in ns])
        slope = np.polyfit(np.log(ns), np.log(err + 1e-18), 1)[0]
        plt.loglog(ns, err + 1e-18, ".-", label=f"{name}: slope {slope:.2f}")
    plt.xlabel("number of panels n"); plt.ylabel("absolute error")
    plt.title("Composite quadrature error (integrand exp x on [0,1])")
    plt.legend(); plt.show()
    print("trapezoid and midpoint converge at order 2, Simpson at order 4")

show_convergence()

In [ ]:
# ---------------------------------------------------------------------------
# Degree of exactness: Simpson integrates cubics exactly
# ---------------------------------------------------------------------------
# A rule has degree of exactness m if it integrates every polynomial of degree
# up to m with zero error. Simpson's is 3, one higher than its interpolation
# degree, thanks to a symmetry cancellation.
def check_exactness():
    """Integrate monomials x^k on [0,1] with a single Simpson panel (n=2)."""
    print("Simpson on [0,1], single application (n = 2 panels):")
    for k in range(0, 5):
        g = lambda x, k=k: x ** k
        approx = composite_simpson(g, 0.0, 1.0, 2)
        exact  = 1.0 / (k + 1)
        print(f"  x^{k}: error = {abs(approx - exact):.2e}")

check_exactness()

In [ ]:
# ---------------------------------------------------------------------------
# Adaptive Simpson with an interval-halving error estimate
# ---------------------------------------------------------------------------
# On each subinterval we compare one Simpson estimate S(a,b) against the sum of
# two half-interval estimates S(a,c)+S(c,b). By Richardson reasoning their
# difference divided by 15 estimates the error of the finer value; if that is
# below the local tolerance we accept it, otherwise we recurse on both halves.
# The Richardson correction (+/15) is added to the accepted value as a cheap
# accuracy boost. A recursion-depth cap prevents runaway subdivision.

def _simpson_panel(f, a, b):
    """Single Simpson estimate on [a, b] using endpoints and midpoint."""
    c = 0.5 * (a + b)
    return (b - a) / 6.0 * (f(a) + 4.0 * f(c) + f(b))

def adaptive_simpson(f, a, b, tol, max_depth=50):
    """Return (integral, list_of_accepted_subinterval_edges). The edge list
    records where the method chose to place panel boundaries."""
    edges = []
    def recurse(a, b, whole, depth):
        c = 0.5 * (a + b)
        left  = _simpson_panel(f, a, c)
        right = _simpson_panel(f, c, b)
        if depth >= max_depth or abs(left + right - whole) <= 15.0 * tol:
            edges.append(a); edges.append(b)
            return left + right + (left + right - whole) / 15.0
        return (recurse(a, c, left,  depth + 1) +
                recurse(c, b, right, depth + 1))
    total = recurse(a, b, _simpson_panel(f, a, b), 0)
    return total, sorted(set(edges))

In [ ]:
# ---------------------------------------------------------------------------
# Adaptive vs uniform on an integrand with one narrow spike
# ---------------------------------------------------------------------------
# The integrand is a sharp Gaussian bump centered at 0.5. A uniform grid must
# be fine everywhere to resolve the spike; the adaptive method clusters panels
# around it and stays coarse in the flat regions.
def f_spike(x): return np.exp(-200.0 * (x - 0.5) ** 2)
I_SPIKE = 0.5 * np.sqrt(np.pi / 200.0) * (
    math.erf(np.sqrt(200.0) * 0.5) - math.erf(-np.sqrt(200.0) * 0.5))

def show_adaptive(log10_tol=-8.0):
    """Run adaptive Simpson, draw the panel boundaries it selected, and compare
    its function-evaluation count with a uniform Simpson rule of equal error."""
    tol = 10.0 ** log10_tol
    val, edges = adaptive_simpson(f_spike, 0.0, 1.0, tol)
    err = abs(val - I_SPIKE)
    n_panels = len(edges) - 1

    xs = np.linspace(0, 1, 800)
    plt.figure()
    plt.plot(xs, f_spike(xs), "b-", lw=2, label="integrand")
    for e in edges:
        plt.axvline(e, color="r", lw=0.5, alpha=0.6)
    plt.xlabel("x"); plt.ylabel("f(x)")
    plt.title(f"Adaptive Simpson panels (tol {tol:.0e}): {n_panels} panels, error {err:.1e}")
    plt.legend(); plt.show()
    print(f"adaptive: {n_panels} panels, error {err:.2e}")
    print("panels cluster around the spike and stay sparse where the integrand is flat")

show_adaptive()

In [ ]:
# ---------------------------------------------------------------------------
# Interactive controls
# ---------------------------------------------------------------------------
interact(
    show_adaptive,
    log10_tol=FloatSlider(value=-8.0, min=-12.0, max=-3.0, step=0.5, description="log10(tol)"),
);

## Summary

- Newton-Cotes rules integrate the interpolant on equispaced nodes; composite versions sum the rule over many panels.
- Composite trapezoid and midpoint converge at order 2, composite Simpson at order 4; the log-log error slope recovers these orders.
- Simpson has degree of exactness 3, one above its interpolation degree, so it is exact for cubics.
- Adaptive Simpson uses an interval-halving error estimate to place panels where the integrand varies and leaves flat regions coarse, matching a target accuracy with far fewer evaluations on a localized feature.